# FSW-Net — Explainable Inference

This notebook lets you classify **images or videos** with FSW-Net and get a
**Claude-powered forensic explanation** backed by visual evidence.

| Cell | What it does | UI equivalent |
|------|-------------|---------------|
| **Setup** | Install deps, imports | App init |
| **Config** | Model path + API key | Settings page |
| **Input** | Choose mode + enter path | File picker + radio buttons |
| **Run** | Execute inference | Submit button |
| **Results** | Show evidence + explanation | Results panel |
| **Batch summary** | Table + chart for folder runs | Dashboard |


---
## Cell 1 — Install & Import

In [16]:
# Install only what may be missing — safe to re-run
import subprocess, sys
def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

pip('anthropic')       # Claude API
pip('ipywidgets')      # interactive widgets
pip('opencv-python')   # video frame extraction
pip('pandas')          # batch summary table

print('All dependencies ready.')

All dependencies ready.


In [17]:
import os, sys
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Image as IPImage
from pathlib import Path
from PIL import Image
import io

from inference_pipeline import FSWInferencePipeline, validate_path

print('Imports OK.')

Imports OK.


---
## Cell 2 — Config

Set your model checkpoint path and Anthropic API key here.

> **UI note:** these map to a Settings/Config page or env-var injection.

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURE THESE BEFORE RUNNING
# ─────────────────────────────────────────────

MODEL_PATH = 'best_fsw_net.pth'          # relative to notebooks/

# API key for the explanation LLM. Read from the environment — never paste a
# key into a notebook: it goes into git history and cannot be removed by
# editing the cell afterwards.
#   Windows :  setx GROQ_API_KEY "your-key"      (reopen the terminal)
#   bash    :  export GROQ_API_KEY="your-key"
ANTHROPIC_API_KEY = os.environ.get("GROQ_API_KEY", "")

CLASS_NAMES = ['ai_generated', 'deepfake', 'edited', 'real']  # must match training order

# ─────────────────────────────────────────────

if not os.path.exists(MODEL_PATH):
    print(f'WARNING: model checkpoint not found at {MODEL_PATH}')
    print('Train FSW-Net first using fsw_novel_detector.ipynb')
else:
    pipeline = FSWInferencePipeline(
        model_path        = MODEL_PATH,
        class_names       = CLASS_NAMES,
        anthropic_api_key = ANTHROPIC_API_KEY or None,
    )
    print('Pipeline ready.')

---
## Cell 3 — Input Selection

Choose your **execution mode** and **input path**.  
The widgets below map directly to UI components:

```
  [○ Single file]  [○ Batch folder]         ← RadioButtons
  [○ Image]  [○ Video]                      ← RadioButtons (conditional)
  Path: [________________________]           ← Text input / file picker
  Explain with AI: [✓]                      ← Checkbox
  [  Validate Path  ]                        ← Button
```

In [ ]:
# ── Widget definitions ───────────────────────────────────────────────────────

w_mode = widgets.RadioButtons(
    options=['Single file', 'Batch folder'],
    description='Mode:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)

w_filetype = widgets.RadioButtons(
    options=['Image', 'Video'],
    description='File type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)

w_path = widgets.Text(
    value='',
    placeholder='Paste full path here …',
    description='Path:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px'),
)

w_explain = widgets.Checkbox(
    value=True,
    description='Generate AI explanation (calls GROQ API)',
    style={'description_width': 'initial'},
)

w_max_explain = widgets.IntSlider(
    value=3, min=1, max=20, step=1,
    description='Max explanations (batch):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'),
)

w_frame_step = widgets.IntSlider(
    value=30, min=5, max=120, step=5,
    description='Frame step (video):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'),
)

out_validate = widgets.Output()

btn_validate = widgets.Button(
    description='Validate Path',
    button_style='info',
    icon='check',
)

# Show/hide widgets depending on mode
def on_mode_change(change):
    if change['new'] == 'Batch folder':
        w_filetype.layout.display   = 'none'
        w_frame_step.layout.display = 'none'
        w_max_explain.layout.display = 'flex'
    else:
        w_filetype.layout.display    = 'flex'
        w_max_explain.layout.display = 'none'
        on_filetype_change({'new': w_filetype.value})

def on_filetype_change(change):
    if change['new'] == 'Video' and w_mode.value == 'Single file':
        w_frame_step.layout.display = 'flex'
    else:
        w_frame_step.layout.display = 'none'

w_mode.observe(on_mode_change, names='value')
w_filetype.observe(on_filetype_change, names='value')

# Validate button callback
def on_validate(b):
    with out_validate:
        clear_output()
        path = w_path.value.strip()
        if not path:
            print(' Please enter a path.')
            return
        if w_mode.value == 'Batch folder':
            mode_key = 'batch_folder'
        elif w_filetype.value == 'Video':
            mode_key = 'single_video'
        else:
            mode_key = 'single_image'
        ok, err = validate_path(path, mode_key)
        if ok:
            print(f' Path valid: {path}')
        else:
            print(f' {err}')

btn_validate.on_click(on_validate)

# Initial visibility
on_mode_change({'new': w_mode.value})
on_filetype_change({'new': w_filetype.value})

# ── Layout ───────────────────────────────────────────────────────────────────
display(
    widgets.VBox([
        widgets.HTML('<h3>Input Selection</h3>'),
        w_mode,
        w_filetype,
        w_path,
        w_explain,
        w_max_explain,
        w_frame_step,
        btn_validate,
        out_validate,
    ])
)

---
## Cell 4 — Run Inference

> **UI note:** this maps to a **Submit / Analyse** button.  
> The `_last_result` variable holds the raw output for the Results cell.

In [ ]:
_last_result = None   # stores the most recent inference result

out_run = widgets.Output()
btn_run = widgets.Button(
    description='Run Analysis',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='200px', height='40px'),
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    description='Progress:',
    bar_style='info',
    layout=widgets.Layout(width='500px', display='none'),
)
lbl_status = widgets.Label('')

def on_run(b):
    global _last_result
    with out_run:
        clear_output()

    path    = w_path.value.strip()
    explain = w_explain.value

    if not path:
        lbl_status.value = ' Please enter a path in the Input Selection cell above.'
        return

    btn_run.disabled = True
    lbl_status.value = 'Running …'

    try:
        if w_mode.value == 'Batch folder':
            progress_bar.layout.display = 'flex'
            progress_bar.max = 100
            progress_bar.value = 0

            def prog(done, total, fname):
                progress_bar.value = int(done / max(total, 1) * 100)
                lbl_status.value = f'{done}/{total} — {fname}'

            _last_result = pipeline.run_batch(
                folder_path   = path,
                explain       = explain,
                max_explain   = w_max_explain.value,
                progress_callback = prog,
            )
            lbl_status.value = f'Done. {_last_result["summary"]["succeeded"]} images processed.'

        elif w_filetype.value == 'Video':
            lbl_status.value = 'Extracting and classifying frames …'
            _last_result = pipeline.run_video(
                video_path  = path,
                frame_step  = w_frame_step.value,
                explain     = explain,
            )
            lbl_status.value = (
                f'Done. {_last_result["total_frames_analyzed"]} frames analysed. '
                f'Verdict: {_last_result["overall_prediction"].upper()}'
            )

        else:  # single image
            lbl_status.value = 'Classifying …'
            _last_result = pipeline.run_single(path, explain=explain)
            lbl_status.value = (
                f'Done. Prediction: {_last_result["prediction"].upper()} '
                f'({_last_result["confidence"]*100:.1f}%)'
            )

    except Exception as exc:
        lbl_status.value = f' Error: {exc}'
        import traceback; traceback.print_exc()
    finally:
        btn_run.disabled = False
        progress_bar.layout.display = 'none'
        print('\n Run Cell 5 to display results.')

btn_run.on_click(on_run)

display(widgets.VBox([
    widgets.HTML('<h3>Run</h3>'),
    widgets.HBox([btn_run, lbl_status]),
    progress_bar,
    out_run,
]))

---
## Cell 5 — Results Display

Renders the prediction, evidence figures, and LLM explanation.

> **UI note:** this entire cell maps to a **Results / Report panel** in the UI.

In [ ]:
# ── Rendering helpers ────────────────────────────────────────────────────────

def _pil_to_widget(img: Image.Image, width: int = 700) -> widgets.Image:
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return widgets.Image(value=buf.getvalue(), format='png',
                         layout=widgets.Layout(width=f'{width}px'))

_RISK_COLOR = {
    'LOW':    '#27ae60',
    'MEDIUM': '#f39c12',
    'HIGH':   '#e74c3c',
}

def _verdict_html(prediction, confidence, risk_level):
    risk_word = risk_level.split()[0]   # 'LOW', 'MEDIUM', 'HIGH'
    color = _RISK_COLOR.get(risk_word, '#888')
    return widgets.HTML(f"""
    <div style='border-left:6px solid {color}; padding:12px 18px;
                background:#f8f9fa; border-radius:6px; margin:10px 0;'>
      <h2 style='margin:0; color:{color};'>{prediction.upper()}</h2>
      <p style='margin:4px 0; font-size:16px;'>Confidence: <b>{confidence*100:.1f}%</b></p>
      <p style='margin:0; color:{color};'>Risk: {risk_level}</p>
    </div>
    """)

def _explanation_html(text):
    paragraphs = ''.join(f'<p style="margin:8px 0;">{p.strip()}</p>'
                         for p in text.split('\n') if p.strip())
    return widgets.HTML(f"""
    <div style='border:1px solid #dee2e6; border-radius:8px;
                padding:16px; background:#fff; margin:10px 0;'>
      <h3 style='margin-top:0; color:#2c3e50;'>Forensic Explanation (Claude)</h3>
      <div style='font-size:14px; line-height:1.65; color:#34495e;'>{paragraphs}</div>
    </div>
    """)

def render_single(result: dict):
    """Render one image result with evidence figures and explanation."""
    clear_output(wait=True)

    display(widgets.HTML(f"<h2>Result: {Path(result.get('path','input')).name}</h2>"))
    display(_verdict_html(result['prediction'], result['confidence'],
                          result.get('risk_level', '')))

    # Probability table
    prob_rows = ''.join(
        f"<tr><td><b>{k}</b></td>"
        f"<td><div style='background:#3498db;height:16px;width:{v*200:.0f}px;'></div></td>"
        f"<td>{v*100:.1f}%</td></tr>"
        for k, v in sorted(result['probs'].items(), key=lambda x: -x[1])
    )
    display(widgets.HTML(f"""
    <table style='border-collapse:collapse;margin:10px 0;font-size:14px;'>
      <tr><th style='text-align:left;padding:4px 12px;'>Class</th>
          <th style='padding:4px 12px;'>Confidence</th>
          <th style='padding:4px 8px;'></th></tr>
      {prob_rows}
    </table>"""))

    # Branch attention
    bw = result.get('branch_weights', {})
    bw_rows = ''.join(
        f"<tr><td><b>{k}</b></td>"
        f"<td><div style='background:#e67e22;height:14px;width:{v*300:.0f}px;'></div></td>"
        f"<td>{v:.3f}</td></tr>"
        for k, v in bw.items()
    )
    display(widgets.HTML(f"""
    <h4 style='margin:12px 0 4px;'>Branch Attention (which forensic signal the model used)</h4>
    <table style='border-collapse:collapse;font-size:13px;'>
      {bw_rows}
    </table>"""))

    # Evidence figures
    if 'figures' in result:
        figs = result['figures']
        display(widgets.HTML('<h3>Visual Evidence</h3>'))
        for title, pil_img in figs.items():
            display(widgets.HTML(f'<h4 style="margin-bottom:4px;">{title.replace("_"," ").title()}</h4>'))
            display(_pil_to_widget(pil_img, width=750))

    # LLM explanation
    if 'explanation' in result:
        display(_explanation_html(result['explanation']))


def render_video(result: dict):
    """Render video inference result."""
    clear_output(wait=True)
    display(widgets.HTML(f"<h2>Video Result: {Path(result['path']).name}</h2>"))

    votes = result['vote_distribution']
    total_frames = result['total_frames_analyzed']\n

    display(_verdict_html(
        result['overall_prediction'],
        result['confidence'],
        result.get('risk_level', '')
    ))

    vote_rows = ''.join(
        f"<tr><td><b>{k}</b></td>"
        f"<td><div style='background:#9b59b6;height:14px;width:{v/total_frames*250:.0f}px'></div></td>"
        f"<td>{v} frames ({v/total_frames*100:.1f}%)</td></tr>"
        for k, v in sorted(votes.items(), key=lambda x: -x[1])
    )
    display(widgets.HTML(f"""
    <h4>Frame-level Vote Distribution ({total_frames} frames sampled)</h4>
    <table style='border-collapse:collapse;font-size:13px;'>{vote_rows}</table>
    <p style='color:#666;font-size:12px;'>
      Explanation uses best frame at {result.get('best_frame_timestamp', '?')}s
    </p>"""))

    if 'figures' in result:
        display(widgets.HTML('<h3>Visual Evidence (Best Representative Frame)</h3>'))
        for title, pil_img in result['figures'].items():
            display(widgets.HTML(f'<h4 style="margin-bottom:4px;">{title.replace("_"," ").title()}</h4>'))
            display(_pil_to_widget(pil_img, width=750))

    if 'explanation' in result:
        display(_explanation_html(result['explanation']))


def render_batch(result: dict):
    """Render batch results — summary table, then expandable per-image results."""
    clear_output(wait=True)
    summary = result['summary']
    results = result['results']

    display(widgets.HTML('<h2>Batch Results</h2>'))
    display(widgets.HTML(f"""
    <div style='display:flex;gap:20px;flex-wrap:wrap;margin-bottom:12px;'>
      <div style='border:1px solid #ccc;border-radius:6px;padding:12px 20px;min-width:140px;'>
        <div style='font-size:28px;font-weight:bold;'>{summary['total']}</div>
        <div style='color:#666;'>Total files</div></div>
      <div style='border:1px solid #27ae60;border-radius:6px;padding:12px 20px;min-width:140px;background:#eafaf1;'>
        <div style='font-size:28px;font-weight:bold;color:#27ae60;'>{summary['counts'].get('real',0)}</div>
        <div style='color:#666;'>Real</div></div>
      <div style='border:1px solid #e74c3c;border-radius:6px;padding:12px 20px;min-width:140px;background:#fdedec;'>
        <div style='font-size:28px;font-weight:bold;color:#e74c3c;'>{summary['ai_risk_count']}</div>
        <div style='color:#666;'>AI / Deepfake</div></div>
      <div style='border:1px solid #f39c12;border-radius:6px;padding:12px 20px;min-width:140px;background:#fef9e7;'>
        <div style='font-size:28px;font-weight:bold;color:#f39c12;'>{summary['counts'].get('edited',0)}</div>
        <div style='color:#666;'>Edited</div></div>
    </div>
    """))

    # Pie chart
    if summary['counts']:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        colors = {'real': '#27ae60', 'deepfake': '#e74c3c',
                  'ai_generated': '#c0392b', 'edited': '#f39c12'}
        c_vals = [colors.get(k, '#888') for k in summary['counts']]
        axes[0].pie(summary['counts'].values(), labels=summary['counts'].keys(),
                    colors=c_vals, autopct='%1.1f%%', startangle=90)
        axes[0].set_title('Prediction Distribution')

        # Confidence distribution
        confs = [r['confidence'] for r in results if 'error' not in r]
        axes[1].hist(confs, bins=20, color='steelblue', edgecolor='white')
        axes[1].set_xlabel('Confidence'); axes[1].set_ylabel('Count')
        axes[1].set_title('Confidence Distribution')
        plt.tight_layout()
        plt.show()

    # Results table
    rows = []
    for r in results:
        if 'error' in r:
            rows.append({'File': Path(r['path']).name, 'Prediction': 'ERROR',
                         'Confidence': '-', 'Risk': r['error'][:60]})
        else:
            rows.append({
                'File':       Path(r['path']).name,
                'Prediction': r['prediction'],
                'Confidence': f"{r['confidence']*100:.1f}%",
                'Risk':       r.get('risk_level','').split('—')[0].strip(),
                'Explained':  'Yes' if 'explanation' in r else 'No',
            })
    df = pd.DataFrame(rows)
    display(df)

    # Detailed results for explained images
    explained = [r for r in results if 'explanation' in r]
    if explained:
        display(widgets.HTML(f'<h3>Detailed Reports ({len(explained)} images)</h3>'))
        for i, r in enumerate(explained):
            display(widgets.HTML(
                f'<h4 style="border-top:1px solid #eee;padding-top:10px;">'
                f'{i+1}. {Path(r["path"]).name}</h4>'
            ))
            display(_verdict_html(r['prediction'], r['confidence'],
                                  r.get('risk_level', '')))
            if 'figures' in r:
                display(_pil_to_widget(r['figures']['decision_analysis'], width=650))
                display(_pil_to_widget(r['figures']['srm_residuals'], width=650))
            display(_explanation_html(r['explanation']))


# ── Render button ────────────────────────────────────────────────────────────

out_results = widgets.Output()
btn_show    = widgets.Button(
    description='Show Results',
    button_style='primary',
    icon='bar-chart',
    layout=widgets.Layout(width='200px', height='40px'),
)

def on_show(b):
    with out_results:
        if _last_result is None:
            clear_output()
            print(' Run Cell 4 first.')
            return
        r = _last_result
        if isinstance(r, dict) and r.get('type') == 'video':
            render_video(r)
        elif isinstance(r, dict) and 'results' in r:
            render_batch(r)
        else:
            render_single(r)

btn_show.on_click(on_show)
display(widgets.VBox([widgets.HTML('<h3>Results</h3>'), btn_show, out_results]))

---
## Cell 6 — Export Report

Save a JSON report and all evidence images to disk.

> **UI note:** maps to a **Download / Export** button.

In [ ]:
import json, datetime

def export_report(result, out_dir='../outputs/reports'):
    """Save JSON report + evidence images. Returns the output directory."""
    ts      = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir = Path(out_dir) / ts
    out_dir.mkdir(parents=True, exist_ok=True)

    def _serialisable(r):
        """Strip PIL images and internal keys for JSON."""
        if isinstance(r, dict):
            return {k: _serialisable(v) for k, v in r.items()
                    if k not in ('figures', '_tensor', '_probs_np',
                                 '_weights_np', '_pred_idx')}
        if isinstance(r, list):
            return [_serialisable(x) for x in r]
        if isinstance(r, np.ndarray):
            return r.tolist()
        return r

    # Save JSON
    with open(out_dir / 'report.json', 'w') as f:
        json.dump(_serialisable(result), f, indent=2)

    # Save figures if present
    def _save_figs(r, prefix=''):
        figs = r.get('figures', {})
        for name, img in figs.items():
            img.save(out_dir / f'{prefix}{name}.png')

    if isinstance(result, dict) and 'results' in result:   # batch
        for i, r in enumerate(result['results']):
            _save_figs(r, prefix=f'img{i:03d}_')
    else:
        _save_figs(result)

    print(f'Report saved to: {out_dir.resolve()}')
    return out_dir


out_export = widgets.Output()
btn_export = widgets.Button(
    description='Export Report',
    button_style='warning',
    icon='download',
)

def on_export(b):
    with out_export:
        clear_output()
        if _last_result is None:
            print('Run inference first.')
            return
        export_report(_last_result)

btn_export.on_click(on_export)
display(widgets.VBox([
    widgets.HTML('<h3>Export</h3>'),
    btn_export,
    out_export,
]))

---
## UI Mapping Guide

When you build the front-end, each widget here maps to a standard UI component:

```
Notebook widget              → UI component
─────────────────────────────────────────────────────────
w_mode  RadioButtons         → Toggle / Tab bar
w_filetype RadioButtons      → Segmented control
w_path  Text                 → FilePicker / drag-drop zone
w_explain Checkbox           → Toggle switch
w_max_explain IntSlider      → Slider or number input
w_frame_step IntSlider       → Slider or number input
btn_validate Button          → "Check" icon button
btn_run Button               → Primary CTA button
progress_bar IntProgress     → Progress bar / spinner
render_single()              → Results panel (card layout)
render_batch()               → Dashboard + expandable rows
render_video()               → Results panel + timeline bar
btn_export Button            → Download button

Back-end calls (UI calls these directly):
  pipeline.run_single(path, explain=True)
  pipeline.run_batch(folder, explain=True, max_explain=N)
  pipeline.run_video(path, frame_step=N, explain=True)
```

The `inference_pipeline.py` module is the **only** import your UI app needs.